### Neural Network For Language Translation

In [21]:
#importing libraries

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import unicodedata
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [22]:
# loading dataset

NUM_EXAMPLES = 20000
data_examples = []
with open("data/deu.txt", "r", encoding="utf-8") as data:
    for line in data.readlines():
        if len(data_examples) < 20000:
            data_examples.append(line)

In [23]:
#sample data
data_examples[0]

'Hi.\tHallo!\tCC-BY 2.0 (France) Attribution: tatoeba.org #538123 (CM) & #380701 (cburgmer)\n'

In [24]:
#separating english and german sentences into different lists
english_list = []
german_list = []
for i in range(len(data_examples)):
    english_sent, german_sent = data_examples[i].split("\t")[:2]
    english_list.append(english_sent)
    german_list.append(german_sent)


In [25]:
#some random english and german sentences

inx = np.random.choice(len(data_examples), 5, replace=True)
for i in inx:
    print(f"English sentence: {english_list[i]}")
    print(f"German sentence: {german_list[i]}")
    print()

English sentence: Can I keep this?
German sentence: Kann ich das behalten?

English sentence: I admired Tom.
German sentence: Ich bewunderte Tom.

English sentence: Lock the door.
German sentence: Schließen Sie die Tür ab.

English sentence: Tom may be out.
German sentence: Tom ist vielleicht draußen.

English sentence: You're the best.
German sentence: Sie sind der Beste.



In [26]:
# These functions preprocess English and German sentences

def unicode_to_ascii(s):
    return ''.join(c for c in unicodedata.normalize('NFD', s) if unicodedata.category(c) != 'Mn')

def preprocess_sentence(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r"ü", 'ue', sentence)
    sentence = re.sub(r"ä", 'ae', sentence)
    sentence = re.sub(r"ö", 'oe', sentence)
    sentence = re.sub(r'ß', 'ss', sentence)
    
    sentence = unicode_to_ascii(sentence)
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    sentence = re.sub(r"[^a-z?.!,']+", " ", sentence)
    sentence = re.sub(r'[" "]+', " ", sentence)
    
    return sentence.strip()

In [27]:
preprocessed_german = list(map(preprocess_sentence, german_list))
preprocessed_english = list(map(preprocess_sentence, english_list))

In [28]:
#adding <start> and <end> tokens for greman list of sentences

token_german=[]
for i in range(len(preprocessed_german)):
    token_german.append("<start> " + preprocessed_german[i] + " <end>")
    
print(token_german[0]+"\n") 

print(preprocessed_german[0])

<start> hallo ! <end>

hallo !


In [29]:
#tokenizing german sentences


tokenizer = Tokenizer(lower=True, char_level=False)

tokenizer.fit_on_texts(token_german)
german_sequence = tokenizer.texts_to_sequences(token_german)

In [30]:
vocabulary = tokenizer.word_index

In [31]:
#samples view

inx = np.random.choice(len(token_german), 5, replace=False)

for i in inx:
    print(preprocessed_english[i])
    print(preprocessed_german[i])
    print(german_sequence[i])
    print(token_german[i])
    print()

he likes sleeping .
ihm gefaellt es zu schlafen .
[1, 119, 244, 7, 17, 256, 2]
<start> ihm gefaellt es zu schlafen . <end>

please follow me .
folgen sie mir bitte .
[1, 488, 6, 18, 63, 2]
<start> folgen sie mir bitte . <end>

look at that .
sieh dir das an .
[1, 408, 48, 8, 32, 2]
<start> sieh dir das an . <end>

tom killed a man .
tom toetete einen mann .
[1, 4, 2949, 36, 189, 2]
<start> tom toetete einen mann . <end>

mary is pregnant .
mary ist schwanger .
[1, 231, 5, 2729, 2]
<start> mary ist schwanger . <end>



In [32]:
#padding german sequences

padded_german_sequences = pad_sequences(german_sequence, padding="post")

In [33]:
#embedding english list
#load embedding module from Tensorflow Hub

embedding_layer = hub.KerasLayer("https://tfhub.dev/google/tf2-preview/nnlm-en-dim128/1", 
                                 output_shape=[128], input_shape=[], dtype=tf.string)

In [ ]:
#testing the layer

embedding_layer(tf.constant(["these", "aren't", "the", "droids", "you're", "looking", "for"])).shape

TensorShape([7, 128])

NOTE:

- As part of data preprocessing we will use a pre-trained English word embedding module from TensorFlow Hub.
- This embedding takes a batch of text tokens in a 1-D tensor of strings as input. It then embeds the separate tokens into a 128-dimensional space.
- This model can also be used as a sentence embedding module. The module will process each token by removing punctuation and splitting on spaces. It then averages the word embeddings over a sentence to give a single embedding vector. However, we will use it only as a word embedding module, and will pass each word in the input sentence as a separate token.


In [36]:
#splitting data into training and validation sets

training_len= int(len(preprocessed_english) - 0.2*(len(preprocessed_english)))

train_english = preprocessed_english[: training_len]
train_german = padded_german_sequences[: training_len]

validation_english = preprocessed_english[training_len:]
validation_german = padded_german_sequences[training_len:]

In [37]:
#loading them into dataset

train_dataset = tf.data.Dataset.from_tensor_slices((train_english, train_german))
validation_dataset = tf.data.Dataset.from_tensor_slices((validation_english, validation_german))

In [38]:
#splitting english sentences into words to embed them

train_dataset = train_dataset.map(lambda english, german : (tf.strings.split(english, sep=" "), german))
validation_dataset = validation_dataset.map(lambda english, german : (tf.strings.split(english, sep=" "), german))

In [40]:
#passing english words through embedding layer created earlier

train_dataset = train_dataset.map(lambda english, german : (embedding_layer(english), german))
validation_dataset = validation_dataset.map(lambda english, german : (embedding_layer(english), german))

In [41]:
#filtering out dataset items if words in english sentence is greater than 13.
train_dataset = train_dataset.filter(lambda english, german : tf.less(tf.shape(english)[0], 13))
validation_dataset = validation_dataset.filter(lambda english, german : tf.less(tf.shape(english)[0], 13))

In [42]:
#adding padding to each english sequence vector

def adjust_dims(english, german):
    h = tf.shape(english)[0]
    english, german = tf.pad(english, [[0, 13-h], [0, 0]]), german
    return english, german

train_dataset = train_dataset.map(adjust_dims)
validation_dataset = validation_dataset.map(adjust_dims)